In [1]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


In [2]:
# Set up database connection
user = 'pgadmin'                                                                #Benutzername für PostgreSQL
password = 'geheim'                                                             #Passwort des Datenbankbenutzers
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers
port = '5432'                                                                   #Port, über den PostgreSQL erreichbar ist
database = 'postgres'                                                           #Name der Datenbank, mit der verbunden wird

# Create Connection URL
db_connection_url = (
    f'postgresql://{user}:{password}@{host}:{port}/{database}'                  #Verbindungsadresse aus den oben definierten Angaben zusammensetzen
)                                                                               

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine erstellen; Verbindung wird erst bei Bedarf geöffnet

# Test database connection
with engine.connect() as connection:                                            #Verbindung öffnen und nach dem Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der aktuellen Datenbank abfragen
    print('Verbundene Datenbank:', result.scalar())                             #Einzelnen Ergebniswert auslesen und anzeigen

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

Verbundene Datenbank: postgres


### Retrieve all rows and all columns from the bicycles table

In [3]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine für die Datenbankverbindung erstellen

# Execute SQL query
df_bicycles = pd.read_sql_query("""
    SELECT *                                                --Alle Spalten auswählen
    FROM bicycles;                                          --Daten aus der Tabelle bicycles abrufen
""", con=engine)                                                                #SQL ausführen und Ergebnis als DataFrame speichern

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

# Show results
display(df_bicycles)                                                            #Ergebnistabelle anzeigen

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


# Aggregatfunktionen COUNT, AVG, MIN, MAX und SUM sowie GROUP BY & HAVING

### Determine the total number of bicycles. Name the result bicycle_count

In [4]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS bicycle_count        --Alle Zeilen zählen und das Ergebnis bicycle_count nennen
    FROM bicycles;                          --Ohne GROUP BY entsteht eine Gesamtanzahl für die Tabelle
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycle_count
0,2000


### Determine the total number of bicycles and the number of bicycles with a known price. Name the results bicycle_count and known_price_count

In [5]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS bicycle_count,               --Alle Zeilen zählen, auch solche mit fehlendem Preis
           COUNT(price) AS known_price_count        --Nur Zeilen mit vorhandenem Preis zählen; ein Preis von 0 zählt mit
    FROM bicycles;                                  --Beide Anzahlen über die gesamte Tabelle berechnen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycle_count,known_price_count
0,2000,2000


### Determine the number of distinct bicycle makes. Name the result make_count

In [6]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(DISTINCT make) AS make_count    --Unterschiedliche Marken zählen; Duplikate und NULL werden nicht mitgezählt
    FROM bicycles;                               --Eine Gesamtanzahl über die gesamte Tabelle berechnen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make_count
0,10


### Determine the number of bicycles per status. Display status and bicycle_count, sorted by bicycle_count in descending order

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT status,                          --Status jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count        --Anzahl Fahrräder innerhalb jeder Statusgruppe zählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    GROUP BY status                         --Fahrräder mit demselben Status zu einer Gruppe zusammenfassen
    ORDER BY bicycle_count DESC;            --Gruppen mit den meisten Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,status,bicycle_count
0,Sold,684
1,Available,672
2,In Maintenance,644


### Determine the number of distinct bicycle models per make. Display make and model_count, sorted by model_count in descending order

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                 --Fahrradmarke jeder Gruppe anzeigen
           COUNT(DISTINCT model) AS model_count  --Unterschiedliche Modelle pro Marke zählen; NULL nicht mitzählen
    FROM bicycles                                --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                --Fahrräder nach Marke gruppieren
    ORDER BY model_count DESC;                   --Marken mit den meisten unterschiedlichen Modellen zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model_count
0,Bianchi,3
1,Cannondale,3
2,Cervelo,3
3,Cube,3
4,Giant,3
5,Merida,3
6,Santa Cruz,3
7,Scott,3
8,Specialized,3
9,Trek,3


### Determine the average bicycle price for each make. Display make and average_price, rounded to two decimal places and sorted by average_price in descending order, with missing averages last

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                    --Fahrradmarke jeder Gruppe anzeigen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnitt berechnen, in NUMERIC umwandeln und auf zwei Stellen runden
    FROM bicycles                                                   --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                                   --Durchschnitt für jede Marke separat berechnen; AVG ignoriert NULL
    ORDER BY average_price DESC NULLS LAST;                         --Höchsten Durchschnitt zuerst anzeigen; fehlende Durchschnitte zuletzt
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,average_price
0,Cube,4603.98
1,Bianchi,4556.95
2,Santa Cruz,4499.56
3,Specialized,4477.58
4,Trek,4417.79
5,Cannondale,4340.02
6,Cervelo,4313.50
7,Giant,4225.80
8,Merida,4218.95
9,Scott,4193.24


### Determine the average bicycle price for each combination of make and model. Display make, model and average_price, rounded to two decimal places. Sort by make and then by average_price, both in descending order

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                    --Fahrradmarke auswählen
           model,                                                   --Fahrradmodell auswählen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnittspreis je Gruppe auf zwei Dezimalstellen runden
    FROM bicycles                                                   --Daten aus der Tabelle bicycles abrufen
    GROUP BY make, model                                            --Für jede Kombination aus Marke und Modell eine Gruppe bilden
    ORDER BY make DESC,                                             --Zuerst nach Marke absteigend sortieren
             average_price DESC;                                    --Innerhalb derselben Marke nach Durchschnittspreis absteigend sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model,average_price
0,Trek,Marlin,4666.03
1,Trek,Domane,4466.53
2,Trek,Emonda,4119.89
3,Specialized,Rockhopper,4742.53
4,Specialized,Roubaix,4504.83
5,Specialized,Allez,4128.30
6,Scott,Aspect,4297.69
7,Scott,Addict,4178.61
8,Scott,Spark,4091.72
9,Santa Cruz,Bronson,4747.68


### Determine how many bicycles per make have a price between 2000 and 4000, including both limits. Display make and bicycle_count, sorted by bicycle_count in descending order

In [11]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                            --Fahrradmarke jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count        --Anzahl Fahrräder pro Marke nach der Filterung zählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000       --Vor dem Gruppieren auf Preise von 2000 bis 4000 inklusive filtern
    GROUP BY make                           --Verbleibende Fahrräder nach Marke gruppieren
    ORDER BY bicycle_count DESC;            --Marken mit den meisten passenden Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,bicycle_count
0,Cannondale,66
1,Giant,64
2,Bianchi,57
3,Specialized,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


### Identify bicycle makes with an average price greater than 3000. Display make and average_price, rounded to two decimal places and sorted by average_price in descending order

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                        --Fahrradmarke jeder Gruppe anzeigen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price       --Durchschnittspreis berechnen und auf zwei Dezimalstellen runden
    FROM bicycles                                                       --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                                       --Fahrräder nach Marke gruppieren
    HAVING AVG(price) > 3000                                            --Gruppen nach der Aggregation filtern; ungerundeten Durchschnitt prüfen
    ORDER BY average_price DESC;                                        --Höchsten Durchschnitt zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,average_price
0,Cube,4603.98
1,Bianchi,4556.95
2,Santa Cruz,4499.56
3,Specialized,4477.58
4,Trek,4417.79
5,Cannondale,4340.02
6,Cervelo,4313.50
7,Giant,4225.80
8,Merida,4218.95
9,Scott,4193.24


### Identify bicycle makes with more than 10 bicycles priced between 2000 and 4000, including both price limits. Display make and bicycle_count, sorted by bicycle_count in descending order

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                --Fahrradmarke jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count            --Anzahl der Fahrräder pro Marke nach der Preisfilterung zählen
    FROM bicycles                               --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000           --Einzelne Zeilen vor der Gruppierung nach Preis filtern
    GROUP BY make                               --Verbleibende Fahrräder nach Marke gruppieren
    HAVING COUNT(*) > 10                        --Nur Gruppen mit mehr als zehn passenden Fahrrädern behalten
    ORDER BY bicycle_count DESC;                --Gruppen mit den meisten passenden Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,bicycle_count
0,Cannondale,66
1,Giant,64
2,Bianchi,57
3,Specialized,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


### Determine the minimum, maximum and average bicycle price. Name the results min_price, max_price and average_price, and round all three values to two decimal places

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT ROUND(CAST(MIN(price) AS NUMERIC), 2) AS min_price,      --Kleinsten vorhandenen Preis ermitteln und auf zwei Stellen runden
           ROUND(CAST(MAX(price) AS NUMERIC), 2) AS max_price,      --Grössten vorhandenen Preis ermitteln und auf zwei Stellen runden
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnitt der vorhandenen Preise auf zwei Stellen runden
    FROM bicycles;                                                  --Ohne GROUP BY über die gesamte Tabelle rechnen; NULL wird ignoriert
""", con=engine)

engine.dispose()
display(df_spickap01)

,min_price,max_price,average_price
0,800.26,7995.9,4382.23


### Determine the total sales revenue and the number of sales. Name the results total_sales_amount and sales_count, and round the total sales revenue to two decimal places

In [15]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT ROUND(CAST(SUM(saleprice) AS NUMERIC), 2) AS total_sales_amount,     --Verkaufspreise addieren und auf zwei Stellen runden; NULL ignorieren
           COUNT(*) AS sales_count                                              --Alle Verkaufsdatensätze zählen
    FROM sales;                                                                 --Über die gesamte Verkaufstabelle rechnen; kein GROUP BY nötig
""", con=engine)

engine.dispose()
display(df_spickap01)

,total_sales_amount,sales_count
0,2158850.0,684
